<a href="https://colab.research.google.com/github/alexhuygen/arcpyOsource/blob/main/bomenhoogte_analyse_qgis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



### Deel 1: QGIS Python Notebook (Inclusief Gebiedsbegrenzing)

Dit script gebruikt een bestaande polygoonlaag om de AHN-data eerst "bij te snijden" (clippen) voordat de berekening start.

In [ ]:
import processing
from qgis.core import QgsProject, QgsRasterLayer, QgsVectorLayer

# --- 1. CONFIGURATIE ---
aoi_naam = "Mijn_Projectgebied"  # Naam van je polygoonlaag in QGIS
dsm_naam = "DSM_AHN"             # Naam van de DSM rasterlaag
dtm_naam = "DTM_AHN"             # Naam van de DTM rasterlaag
bag_naam = "BAG_Panden"          # Naam van de BAG vectorlaag
output_map = "C:/Temp/"

# Haal de lagen op uit het project
aoi_layer = QgsProject.instance().mapLayersByName(aoi_naam)[0]
dsm_layer = QgsProject.instance().mapLayersByName(dsm_naam)[0]
dtm_layer = QgsProject.instance().mapLayersByName(dtm_naam)[0]
bag_layer = QgsProject.instance().mapLayersByName(bag_naam)[0]

# --- 2. CLIP RASTERS OP BASIS VAN POLYGOON ---
def clip_raster(layer, mask, output_name):
    output_path = output_map + output_name + ".tif"
    params = {
        'INPUT': layer,
        'MASK': mask,
        'NODATA': -9999,
        'TARGET_CRS': layer.crs(),
        'OUTPUT': output_path
    }
    processing.run("gdal:cliprasterbymasklayer", params)
    return QgsRasterLayer(output_path, output_name)

print("Bezig met clippen van AHN data...")
dsm_clipped = clip_raster(dsm_layer, aoi_layer, "DSM_Project_Clipped")
dtm_clipped = clip_raster(dtm_layer, aoi_layer, "DTM_Project_Clipped")

# --- 3. BAG MASKER VOORBEREIDEN ---
print("Bezig met maken van gebouw-masker...")
bag_raster_path = output_map + "bag_mask.tif"
# We maken een raster waar gebouwen waarde 1 hebben, de rest 0
params_rasterize = {
    'INPUT': bag_layer,
    'BURN_VALUE': 1,
    'USE_Z': False,
    'EXTENT': dsm_clipped.extent(),
    'WIDTH': dsm_clipped.width(),
    'HEIGHT': dsm_clipped.height(),
    'OUTPUT': bag_raster_path
}
processing.run("gdal:rasterize", params_rasterize)
bag_mask = QgsRasterLayer(bag_raster_path, "BAG_Mask")

# --- 4. BEREKENING (CHM) ---
print("Bezig met finale berekening...")
final_output = output_map + "Definitieve_Boomhoogte.tif"
# Formule: (DSM - DTM) en zet alles op 0 waar een gebouw (bag=1) staat of hoogte < 0.5m
# (A - B) * ( (A - B) > 0.5 ) * (C = 0)
params_calc = {
    'INPUT_A': dsm_clipped,
    'BAND_A': 1,
    'INPUT_B': dtm_clipped,
    'BAND_B': 1,
    'INPUT_C': bag_mask,
    'BAND_C': 1,
    'FORMULA': '(A-B) * ((A-B)>0.5) * (C==0)',
    'OUTPUT': final_output
}
processing.run("gdal:rastercalculator", params_calc)

# Resultaat toevoegen aan QGIS
QgsProject.instance().addMapLayer(QgsRasterLayer(final_output, "Resultaat Boomhoogte"))
print("Analyse voltooid!")

---

### Deel 2: Handleiding (Inhoud voor Word Document)

**Titel: Werkinstructie Bomenanalyse met Gebiedsbegrenzing**

**1. Gebruik van Projectgebied (AOI)**
Om de verwerkingstijd te verkorten, wordt er gewerkt met een polygoon die het projectgebied definieert. Alle invoerlagen (DSM, DTM en BAG) worden op basis van deze polygoon bijgesneden.

**2. Benodigde Lagen**

* **Project_Grens:** Een polygoonlaag die de omvang van de analyse bepaalt.
* **DSM & DTM:** Rasterbestanden (bijv. AHN4/5).
* **BAG_Panden:** Vectorlaag met gebouwencontouren.

**3. Workflow in Stappen**

1. **Clippen:** Gebruik de tool *Clip Raster by Mask Layer*. Selecteer de AHN-raster als 'Input' en je projectpolygoon als 'Mask Layer'. Doe dit voor zowel de DSM als de DTM.
2. **Gebouwen Rasteriseren:** Zet de BAG-panden om naar een raster met de tool *Rasterize (Vector to Raster)*. Zorg dat de resolutie en 'extent' exact gelijk zijn aan de gecleppte AHN-data.
3. **Raster Calculator:** Gebruik de calculator om de formule `(DSM - DTM) * (BAG_Mask == 0)` uit te voeren.
* *Resultaat:* Een kaart waarop gebouwen zijn 'uitgestanst' en alleen de hoogtes van bomen en struiken overblijven.


4. **Hoogte-Filter:** Filter waarden onder de 0,5 meter weg om ruis (zoals geparkeerde auto's of straatmeubilair) te verwijderen.

**4. Visualisatie**
Stel de symbologie in op *Singleband Pseudocolor* met de volgende klassen voor een helder overzicht:

* 0 - 2m: Struweel / Lage beplanting (Lichtgroen)
* 2 - 10m: Middelhoge bomen (Groen)
* 10 - 20m: Volwassen bomen (Donkergroen)
* > 20m: Zeer hoge bomen (Zeer donkergroen)

**5. CAD Integratie**
Exporteer het resultaat als GeoTIFF (EPSG:28992 - RD New). In AutoCAD Map 3D kan dit bestand via een FDO-connectie of het commando `MAPIINSERT` direct op de juiste plek in de tekening worden gezet.